In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("hour.csv")
data["dteday"] = pd.to_datetime(data["dteday"])
data["day"] = data["dteday"].dt.day
data["month"] = data["dteday"].dt.month
data["year"] = data["dteday"].dt.year

data = data.drop(columns=["dteday"])

In [ ]:
X = data.drop(columns=["cnt"])
y = data["cnt"]

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "rmse": make_scorer(lambda y_true, y_pred:
                        np.sqrt(mean_squared_error(y_true, y_pred)),
                        greater_is_better=False),
    "mae": make_scorer(mean_absolute_error, greater_is_better=False)
}

In [ ]:
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)

In [ ]:
subag = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=15),
    n_estimators=200,
    max_samples=0.7,
    bootstrap=False,
    random_state=42
)

In [ ]:
xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

In [ ]:
def evaluate_model(model, name):
    cv_results = cross_validate(model, X, y,
                                cv=kf,
                                scoring=scoring,
                                return_train_score=False)

    rmse_mean = -cv_results["test_rmse"].mean()
    rmse_std  = cv_results["test_rmse"].std()

    mae_mean = -cv_results["test_mae"].mean()
    mae_std  = cv_results["test_mae"].std()

    return {
        "Model": name,
        "RMSE_mean": rmse_mean,
        "RMSE_std": rmse_std,
        "MAE_mean": mae_mean,
        "MAE_std": mae_std
    }

In [ ]:
results = []

results.append(evaluate_model(rf, "RandomForest"))
results.append(evaluate_model(subag, "Subagging"))
results.append(evaluate_model(xgb, "XGBoost"))

results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("cv_regression_results.csv", index=False)

          Model  RMSE_mean  RMSE_std  MAE_mean   MAE_std
0  RandomForest   2.725582  0.543761  0.950882  0.039987
1     Subagging   2.814948  0.547802  1.039156  0.041761
2       XGBoost   4.968879  0.274313  2.928878  0.086089


In [ ]:
best_model = xgb
best_model.fit(X, y)
preds = best_model.predict(X)

final_df = pd.DataFrame({
    "ActualCnt": y,
    "PredictedCnt": preds
})
final_df.to_csv("final_predictions.csv", index=False)

Random Forest generalizes best, achieving the lowest RMSE and MAE. It reduces variance through averaging many uncorrelated trees while keeping bias low, giving the best bias–variance tradeoff compared to Subagging and XGBoost.

In [ ]:
importances = best_model.feature_importances_

feat_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

top8 = feat_df.head(8)
print("Top 8 Features:\n", top8)

Top 8 Features:
        Feature  Importance
14  registered    0.548562
17        year    0.215031
13      casual    0.133235
7   workingday    0.041922
4           hr    0.039870
0      instant    0.011167
6      weekday    0.005398
9         temp    0.001385
